# Exp8a · Xu-inspired dual-adjacency GCN with a cortical-surface anatomical layer (amendment A2)

**Why this notebook exists.** On the recorded analysis atlas (lobes12) the original Exp 8 anatomical layer
(`lobe_hemisphere`: regions of the same lobe and hemisphere) is the zero matrix, because the 12 regions ARE the
lobe x hemisphere groups; every original Exp 8 unit failed before training. Amendment A2 (post-design, pre-result;
methodology log entry 60) redefines layer 2 as the fixed **cortical-surface contiguity** of the atlas regions
(fsaverage mesh, within a hemisphere): a data- and label-independent anatomical-neighbourhood prior, not measured
white-matter connectivity. Layer 1 is unchanged: the window's frequency-specific PLV or |ciPLV|. This is an
adaptation inspired by Xu et al. (2025), not a reproduction.

Config `configs/final/amendments/exp8a_xu_dual_gcn_surface.yaml` (`exp8a_xu_dual_gcn_surface`, 10 variants; `alt_bilateral_lobes` is an alternative
anatomical prior: each lobe linked to its contralateral homologue):

- `plv__psd_de_strength`: layer 1 functional, layer 2 anatomical (surface_contiguity)
- `ciplv__psd_de_strength`: layer 1 functional, layer 2 anatomical (surface_contiguity)
- `plv__psd`: layer 1 functional, layer 2 anatomical (surface_contiguity)
- `ciplv__psd`: layer 1 functional, layer 2 anatomical (surface_contiguity)
- `plv__de`: layer 1 functional, layer 2 anatomical (surface_contiguity)
- `ciplv__de`: layer 1 functional, layer 2 anatomical (surface_contiguity)
- `plv__psd_de_strength__abl_functional_both`: layer 1 functional, layer 2 functional
- `plv__psd_de_strength__abl_anatomical_both`: layer 1 anatomical, layer 2 anatomical (surface_contiguity)
- `plv__psd_de_strength__alt_bilateral_lobes`: layer 1 functional, layer 2 anatomical (lobe_bilateral)
- `plv__psd_de_strength__abl_shared_weights`: layer 1 functional, layer 2 anatomical (surface_contiguity), weights shared across bands

**Isolation.** This notebook checks out the amendment's own code pin
(`<outputs>/FINAL_v1/amendments/A2_exp8_surface/CODE_COMMIT.txt`) and writes only to `<outputs>/FINAL_v1/amendments/A2_exp8_surface/`.
The release pin, Exp 0-7, their results and the failed / partial units of the original Exp 8 are not touched or read
as checkpoints. Stores, QC reports and the atlas / graph decisions are read only. Same protocol (`gnn_fixed60_v1`),
folds and seeds; development CV only (the holdout is not run here).

**Holdout policy.** The 13-participant holdout was inspected during earlier experiments (LX / WX suites), so it is no longer an untouched test set. In this release it is evaluated once per final configuration and reported descriptively; it is never used to select architectures, graph constructions, k, atlas resolution, QC thresholds or any other setting (selection reads development CV results only).

**How to use:** *Runtime → Run all*. Allow Google Drive access when asked. If Colab disconnects, open the notebook
again and *Run all* again: finished work is kept on Google Drive and skipped. Everything is written to
`<outputs>/FINAL_v1/` on Drive (never to the temporary Colab disk); the validated graph stores and all earlier
results are only read.
 Use a **GPU runtime** (T4).

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_CODE = False   # leave False. True only to move THIS AMENDMENT's code pin deliberately
                      # (<outputs>/FINAL_v1/amendments/A2_exp8_surface/CODE_COMMIT.txt; the release pin is never used or moved here).
BRANCH = 'claude/unified-final-framework'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py', '--amendment', 'A2'] + (['--update-code'] if UPDATE_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh as _sh

def sh(cmd, cwd=None):  # every script runs with the file backend: Colab's inline backend breaks figures in scripts
    os.environ['MPLBACKEND'] = 'Agg'
    return _sh(cmd, cwd)

from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
FINAL = OUT + '/FINAL_v1'
print('release outputs on Drive:', FINAL)
A2_ROOT = FINAL + '/amendments/A2_exp8_surface'

## 1. Numerical environment (exactly that of the completed Exp 7 units)

Exp 8a is compared with Exp 0-7, so it runs with the Python and library versions recorded in the code identity of a
completed Exp 7 unit (python, numpy, scipy, pandas, scikit-learn, torch). Only these versions are compared: the git
commit and source hash of Exp 8a are its own and are NOT expected to match Exp 7. If this Colab kernel differs, the
environment is restored automatically in `/content/restore<python digits>` (e.g. `/content/restore31315`; built with
`uv`, a few minutes, on the temporary Colab disk only) and verified; every `scripts/final.py` call below uses that
interpreter (`PY`). Leave `ACCEPT_ENVIRONMENT_DIFFERENCE = False`: a difference that cannot be removed stops here.

In [ ]:
ACCEPT_ENVIRONMENT_DIFFERENCE = False   # True only after reviewing a reported difference (it is then recorded)
from adgraph import numerical_env
REFERENCE = numerical_env.reference_libraries(OUT, 'exp7_earm_gnn')   # recorded by a completed Exp 7 unit
PY = numerical_env.ensure_interpreter(REFERENCE, repo='/content/repo', base='/content',
                                      accept_difference=ACCEPT_ENVIRONMENT_DIFFERENCE, run=sh)
NOW = numerical_env.interpreter_versions(PY, REFERENCE)
DIFF = numerical_env.differences(REFERENCE, NOW)
assert ACCEPT_ENVIRONMENT_DIFFERENCE or not DIFF, f'numerical environment differs from Exp 7: {DIFF}'
print('scripts run with', PY, NOW)

## 2. Amendment check: pins, namespace, decisions, stores and QC (nothing is trained)

In [ ]:
sh(f"{PY} scripts/final.py amendment-check --amendment A2 --atlas selected"
   + (" --accept-environment-difference" if ACCEPT_ENVIRONMENT_DIFFERENCE else ""), cwd='/content/repo')

import json, os, subprocess, yaml
def _pin(path):
    return next(ln.strip() for ln in open(path) if ln.strip() and not ln.startswith('#'))
RECORD = yaml.safe_load(open('/content/repo/configs/final/amendments.yaml'))['A2']
MANIFEST = json.load(open(A2_ROOT + '/manifest.json'))
RELEASE_PIN_FILE, A2_PIN_FILE = FINAL + '/CODE_COMMIT.txt', A2_ROOT + '/CODE_COMMIT.txt'
HEAD = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd='/content/repo', text=True).strip()
ATLAS_NAME = json.load(open(FINAL + '/decisions/atlas.json'))['atlas']
GRAPH_NAME = json.load(open(FINAL + '/decisions/graph_construction.json'))['selected']
assert MANIFEST['amendment'] == 'A2' and RECORD['namespace'].endswith('A2_exp8_surface'), 'not amendment A2'
assert ATLAS_NAME == 'lobes12', f'recorded atlas is {ATLAS_NAME}, amendment A2 is defined for lobes12'
assert GRAPH_NAME == 'dense', f'recorded graph construction is {GRAPH_NAME}, expected dense'
assert os.path.realpath(A2_PIN_FILE) != os.path.realpath(RELEASE_PIN_FILE), 'pins must be separate files'
assert _pin(RELEASE_PIN_FILE) == RECORD['base_release_pin'] == MANIFEST['base_release_pin'], 'release pin moved'
assert _pin(A2_PIN_FILE) == HEAD == MANIFEST['amendment_commit'], 'the code is not the amendment pin'
assert _pin(A2_PIN_FILE) != _pin(RELEASE_PIN_FILE), 'the amendment must not run at the release pin'
assert ACCEPT_ENVIRONMENT_DIFFERENCE or MANIFEST['environment_ok'], MANIFEST['library_differences']
print('release pin  ', _pin(RELEASE_PIN_FILE), RELEASE_PIN_FILE)
print('amendment pin', _pin(A2_PIN_FILE), A2_PIN_FILE)
print('atlas', ATLAS_NAME, '| graph', GRAPH_NAME, '| namespace', A2_ROOT)

## 3. The anatomical layer of the recorded atlas (fixed; from the atlas only)

In [ ]:
import numpy as np, pandas as pd
from adgraph.models.xu_dual import anatomical_adjacency
from adgraph.source.atlas import region_names
names = region_names(ATLAS_NAME)
A = anatomical_adjacency(names, 'surface_contiguity')
print(ATLAS_NAME, len(names), 'regions,', int(A.sum() // 2), 'surface edges, symmetric', bool((A == A.T).all()))
display(pd.DataFrame(A.astype(int), index=names, columns=names))

## 4. Contexts, status, run (resumable; this amendment experiment only)

In [ ]:
CONTEXTS = [('lcmv', 'B'), ('wmne', 'B'), ('lcmv', 'A'), ('wmne', 'A')]     # (inverse, window mode), in this order
ERRORS = {}
for inverse, mode in CONTEXTS:
    print(f'==== exp8a | {inverse} | mode {mode}')
    sh(f"{PY} scripts/final.py status --amendment A2 --exp exp8a --inverse {inverse} --mode {mode} "
       f"--atlas selected --graph selected", cwd='/content/repo')
    try:
        sh(f"{PY} scripts/final.py run --amendment A2 --exp exp8a --inverse {inverse} --mode {mode} "
           f"--atlas selected --graph selected", cwd='/content/repo')
    except RuntimeError as exc:
        ERRORS[(inverse, mode)] = exc
        print(f'{inverse} / mode {mode}: some units failed; details after the results')

## 5. Results and diagnostics of THIS amendment only (saved outputs; diagnostic figures)

In [ ]:
import glob, json
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.max_rows', 400, 'display.width', 250, 'display.max_columns', 40)

def latest(pattern):
    found = sorted(glob.glob(pattern))
    return found[-1] if found else None
EXPECTED_VARIANTS = ['ciplv__de', 'ciplv__psd', 'ciplv__psd_de_strength', 'plv__de', 'plv__psd', 'plv__psd_de_strength', 'plv__psd_de_strength__abl_anatomical_both', 'plv__psd_de_strength__abl_functional_both', 'plv__psd_de_strength__abl_shared_weights', 'plv__psd_de_strength__alt_bilateral_lobes']
assert len(EXPECTED_VARIANTS) == 10
for inverse, mode in CONTEXTS:
    sh(f"{PY} scripts/final.py report --amendment A2 --exp exp8a --inverse {inverse} --mode {mode} "
       f"--atlas selected --graph selected", cwd='/content/repo')
    rep = latest(f"{A2_ROOT}/results/{inverse}/mode_{mode}/{ATLAS_NAME}/{GRAPH_NAME}/exp8a_xu_dual_gcn_surface/_reports/report_*")
    if (inverse, mode) in ERRORS:
        continue
    assert rep, f'no report of exp8a_xu_dual_gcn_surface for {inverse} / mode {mode} under {A2_ROOT}'
    print('====', inverse, 'mode', mode, rep)
    t = pd.read_csv(rep + '/results.csv')
    assert sorted(t['variant'].unique()) == EXPECTED_VARIANTS, sorted(t['variant'].unique())
    assert set(t['setting']) == {'xu_dev_cv'}, 'development CV only'
    cols = ['variant', 'setting', 'auc', 'auc_ci_low', 'auc_ci_high', 'mean_fold_auc', 'min_fold_auc',
            'max_fold_auc', 'balanced_accuracy', 'sensitivity', 'specificity', 'tn', 'fp', 'fn', 'tp']
    display(t[[c for c in cols if c in t]].round(3))
    for png in sorted(glob.glob(rep + '/*.png')):
        display(Image(filename=png))
assert _pin(RELEASE_PIN_FILE) == RECORD['base_release_pin'], 'release pin moved'
if ERRORS:
    raise RuntimeError(f'units did not complete in {sorted(ERRORS)}; completed units are saved. Run all again.')